# Name That LLM — offline inference template

Attach the ZIP produced by `python -m src.pipeline package-kaggle` and the competition input dataset to this Kaggle notebook. Update the two input paths below. Disable Internet in the notebook settings before running. The output uses the repository's observed generic `id,model` schema; check the competition rules and sample submission before submitting.

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, zipfile

# Replace these with the paths shown for your attached Kaggle input datasets.
MODEL_PACKAGE = Path('/kaggle/input/name-that-llm-model/model_package.zip')
TEST_CSV = Path('/kaggle/input/competition-data/test.csv')
OUTPUT = Path('/kaggle/working/submission.parquet')

assert MODEL_PACKAGE.is_file(), f'Model package not found: {MODEL_PACKAGE}'
assert TEST_CSV.is_file(), f'Test CSV not found: {TEST_CSV}'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

PACKAGE_ROOT = Path('/kaggle/working/nlm_offline_package')
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(MODEL_PACKAGE) as package:
    package.extractall(PACKAGE_ROOT)
sys.path.insert(0, str(PACKAGE_ROOT))

from src.inference import export_submission
manifest = json.loads((PACKAGE_ROOT / 'manifest.json').read_text())
result = export_submission(manifest['model'], PACKAGE_ROOT / 'manifest.json', TEST_CSV, OUTPUT)
submission = __import__('pandas').read_parquet(OUTPUT)
assert submission.columns.tolist() == ['id', 'model']
assert submission['id'].astype(str).tolist() == __import__('pandas').read_csv(TEST_CSV, dtype={'id': 'string'})['id'].astype(str).tolist()
assert submission['model'].isin(manifest['classes']).all()
display(submission.head())
print({'rows': len(submission), 'output': str(OUTPUT), 'package': result['checkpoint']})